> # Machine Learning Algorithms — Theory, Intuition & scikit-learn Implementation

This notebook covers the most-used ML algorithms, grouped by data type:

**Structured / Tabular Data**
1. Linear Regression
2. Logistic Regression
3. Random Forest
4. XGBoost (Extreme Gradient Boosting)
5. LightGBM (Light Gradient Boosting Machine)

**Unstructured Data (Text, Images, Audio)**
6. CNN — Convolutional Neural Networks
7. Transformers (BERT, GPT architectures)

**Unsupervised Clustering**
8. K-Means Clustering

For each algorithm: **Theory → Intuition → Real-world use cases → Code → Evaluation.**


In [1]:
# ==============================================================
# Setup — install gradient boosting libraries if missing
# ==============================================================
import sys, subprocess
for pkg in ['xgboost', 'lightgbm']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)
print('Ready.')

Ready.


---
# 1️⃣ Linear Regression

### Theory
Linear Regression models the target as a **weighted sum of features**:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$$

Weights are found by minimizing the **Mean Squared Error (MSE)**:

$$\text{MSE} = \frac{1}{m}\sum_{i=1}^{m}(y_i - \hat{y}_i)^2$$

This has a closed-form solution (Normal Equation) or can be solved by Gradient Descent. Regularized variants: **Ridge** (L2 — shrinks weights) and **Lasso** (L1 — drives weak weights to exactly 0, i.e., automatic feature selection).

### Intuition
Imagine plotting house size vs price and drawing the single straight line that stays as close as possible to all points. Each weight answers: *"if this feature increases by 1 unit, how much does the prediction change, holding everything else constant?"* — that's why it is the most **interpretable** model in existence.

### Real-world use cases
- 🏠 **House price prediction** (Zillow-style estimates)
- 📈 **Sales & demand forecasting** as a strong baseline
- 💊 **Medical**: modeling drug dosage vs response
- 📉 **Finance**: CAPM beta estimation, trend estimation
- Any problem where **explaining the "why"** matters as much as accuracy


In [2]:
# ==============================================================
# 1. LINEAR REGRESSION — scikit-learn
# ==============================================================
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# Synthetic regression dataset: 5 informative features + noise
X, y = make_regression(n_samples=800, n_features=8, n_informative=5, noise=15, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_tr, y_tr)                              # learns weights w and intercept w0
y_pred = lr.predict(X_te)

print(f'R²   : {r2_score(y_te, y_pred):.3f}')          # % of variance explained (1.0 = perfect)
print(f'RMSE : {mean_squared_error(y_te, y_pred) ** 0.5:.2f}')  # error in target units
print(f'Learned weights: {lr.coef_.round(1)}')          # note: uninformative features get ~0 weight

# Regularized versions — compare how Lasso zeroes-out useless features
ridge = Ridge(alpha=1.0).fit(X_tr, y_tr)        # L2: shrink all weights smoothly
lasso = Lasso(alpha=1.0).fit(X_tr, y_tr)        # L1: hard-zero weak weights
print(f'\nLasso weights : {lasso.coef_.round(1)}   <- exact zeros = features eliminated')

# Visual: predicted vs actual — points on the diagonal = good predictions
plt.figure(figsize=(5, 4))
plt.scatter(y_te, y_pred, alpha=0.4)
plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--')
plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title('Linear Regression: Predicted vs Actual')
plt.show()

R²   : 0.944
RMSE : 15.59
Learned weights: [-0.7 10.4 65.4  0.5 27.3 13.1  0.2  0.3]

Lasso weights : [-0.   9.4 64.4  0.  26.1 12.   0.   0. ]   <- exact zeros = features eliminated


---
# 2️⃣ Logistic Regression

### Theory
Despite the name, this is a **classification** algorithm. It passes the linear combination through the **sigmoid function** to squash output into a probability [0, 1]:

$$P(y=1\mid x) = \sigma(w^Tx) = \frac{1}{1 + e^{-w^Tx}}$$

Weights are learned by maximizing likelihood ⇔ minimizing **Log Loss (binary cross-entropy)**:

$$L = -\frac{1}{m}\sum \left[y\log\hat{p} + (1-y)\log(1-\hat{p})\right]$$

The **decision boundary** (`p = 0.5`) is a straight line/hyperplane. Coefficients are interpretable as **log-odds**: `exp(w_i)` = odds multiplier per unit increase of feature *i*.

### Intuition
Think of a smooth on/off dimmer instead of a hard switch. Far on one side of the boundary → probability near 0; far on the other side → near 1; near the boundary → uncertain ~0.5. The model literally learns *"how strongly does each feature push toward class 1?"*

### Real-world use cases
- 💳 **Credit scoring / loan default** (banks love it — regulators demand interpretability)
- 📧 **Spam detection** (classic baseline)
- 🏥 **Disease risk prediction** (e.g., probability of diabetes given clinical measurements)
- 📱 **Click-through-rate (CTR) prediction** in online ads — still used at massive scale
- 🔄 **Customer churn** probability


In [3]:
# ==============================================================
# 2. LOGISTIC REGRESSION — scikit-learn
# ==============================================================
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score, RocCurveDisplay)

# Real medical dataset: classify tumors as malignant/benign from 30 measurements
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scaling matters for logistic regression (gradient-based + regularized)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
clf.fit(X_tr, y_tr)

y_pred  = clf.predict(X_te)                      # hard class labels (threshold 0.5)
y_proba = clf.predict_proba(X_te)[:, 1]          # calibrated probabilities — the real output!

print(f'Accuracy : {accuracy_score(y_te, y_pred):.3f}')
print(f'Precision: {precision_score(y_te, y_pred):.3f}   (of predicted positives, how many were right?)')
print(f'Recall   : {recall_score(y_te, y_pred):.3f}   (of actual positives, how many did we catch?)')
print(f'F1       : {f1_score(y_te, y_pred):.3f}   (harmonic mean of P & R)')
print(f'ROC-AUC  : {roc_auc_score(y_te, y_proba):.3f}   (ranking quality across ALL thresholds)')
print('\nConfusion matrix:\n', confusion_matrix(y_te, y_pred))

RocCurveDisplay.from_predictions(y_te, y_proba)
plt.title('ROC Curve — Logistic Regression'); plt.show()

Accuracy : 0.982
Precision: 0.986   (of predicted positives, how many were right?)
Recall   : 0.986   (of actual positives, how many did we catch?)
F1       : 0.986   (harmonic mean of P & R)
ROC-AUC  : 0.995   (ranking quality across ALL thresholds)

Confusion matrix:
 [[41  1]
 [ 1 71]]


---
# 3️⃣ Random Forest

### Theory
A Random Forest is an **ensemble of decision trees** built with two sources of randomness:
1. **Bagging (Bootstrap Aggregating):** each tree trains on a random sample (with replacement) of the rows.
2. **Feature randomness:** at each split, only a random subset of features is considered.

Prediction = **majority vote** (classification) or **average** (regression) across all trees.

**Why it works — variance reduction:** a single deep tree has low bias but *huge* variance (it memorizes noise). Averaging many *decorrelated* trees cancels their individual errors: $\text{Var}(\bar{X}) \approx \rho\sigma^2 + \frac{1-\rho}{n}\sigma^2$. The randomness lowers correlation ρ between trees, making the average dramatically more stable.

### Intuition
**Wisdom of the crowd.** Ask 500 slightly-different experts (each saw different customers and considers different criteria) and take a vote. Individually noisy; collectively remarkably accurate. Bonus: no scaling needed, handles non-linearities and interactions automatically, and gives feature importances for free.

### Real-world use cases
- 🏦 **Fraud detection** in banking transactions
- 🏥 **Disease diagnosis** from lab results / patient records
- 🛒 **Customer churn & credit risk** modeling
- 🌾 **Remote sensing**: land-cover classification from satellite bands
- ⚙️ **Predictive maintenance**: will this machine fail soon?


In [4]:
# ==============================================================
# 3. RANDOM FOREST — scikit-learn
# ==============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

# Harder synthetic tabular problem with non-linear structure
X, y = make_classification(n_samples=2000, n_features=20, n_informative=8,
                           n_redundant=4, flip_y=0.05, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(
    n_estimators=300,        # number of trees — more = more stable (diminishing returns)
    max_depth=None,          # let trees grow deep; the ensemble averages out overfitting
    max_features='sqrt',     # features considered per split — the decorrelation knob
    min_samples_leaf=2,      # small regularization on leaf size
    n_jobs=-1,               # use all CPU cores
    random_state=42,
    oob_score=True,          # FREE validation: each tree tested on rows it never saw
)
rf.fit(X_tr, y_tr)

print(f'Test accuracy       : {rf.score(X_te, y_te):.3f}')
print(f'Out-of-Bag accuracy : {rf.oob_score_:.3f}   <- built-in validation, no data wasted')

# Feature importance: how much each feature reduced impurity across all splits
imp = pd.Series(rf.feature_importances_, index=[f'f{i}' for i in range(20)]).sort_values()
imp.tail(10).plot(kind='barh', figsize=(6, 4), title='Random Forest — Top 10 Feature Importances')
plt.tight_layout(); plt.show()

Test accuracy       : 0.845
Out-of-Bag accuracy : 0.839   <- built-in validation, no data wasted


---
# 4️⃣ XGBoost — Extreme Gradient Boosting

### Theory
**Boosting** builds trees **sequentially** — each new tree is trained to fix the errors (more precisely, the **gradients of the loss**) left by the ensemble so far:

$$\hat{y}^{(t)} = \hat{y}^{(t-1)} + \eta \cdot f_t(x)$$

where η is the **learning rate** (shrinkage) and each $f_t$ fits the negative gradient (≈ residuals). XGBoost adds engineering & math upgrades over plain gradient boosting:
- **Second-order optimization** (uses both gradient *and* hessian of the loss)
- **Built-in regularization** (L1/L2 on leaf weights + `gamma` minimum split gain)
- **Native handling of missing values** (learns the best default direction per split)
- Column/row subsampling, parallelized split finding, cache-aware design

**Boosting vs Bagging:** Random Forest reduces *variance* by averaging independent trees; Boosting reduces *bias* by chaining dependent, corrective trees.

### Intuition
A student takes a practice exam (tree 1), then a tutor writes the *next* practice set focused exactly on the questions they got wrong (tree 2), and so on. Each step is small (learning rate) so the student never overreacts to a single mistake. After hundreds of corrective rounds, performance is exceptional.

### Real-world use cases
- 🏆 **Kaggle & industry king of tabular data** — the default winning model for years
- 💳 **Credit default & fraud detection** at banks and fintechs
- 📊 **Ad ranking / CTR prediction**
- 🛍️ **Sales forecasting & customer lifetime value**
- 🚕 **ETA prediction** (ride-share arrival times)


In [5]:
# ==============================================================
# 4. XGBOOST — scikit-learn compatible API
# ==============================================================
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,        # boosting rounds (trees) — pairs with learning_rate
    learning_rate=0.05,      # smaller = slower, better generalization (needs more trees)
    max_depth=4,             # boosted trees are kept SHALLOW (weak learners)
    subsample=0.8,           # row sampling per tree  -> regularization
    colsample_bytree=0.8,    # feature sampling per tree -> regularization
    reg_lambda=1.0,          # L2 regularization on leaf weights
    eval_metric='logloss',
    early_stopping_rounds=30,  # stop when validation stops improving -> auto-tunes n_estimators
    random_state=42,
)

# Early stopping requires a validation set to watch
X_tr2, X_val, y_tr2, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)
xgb.fit(X_tr2, y_tr2, eval_set=[(X_val, y_val)], verbose=False)

print(f'Best iteration found by early stopping: {xgb.best_iteration}')
print(f'Test accuracy: {xgb.score(X_te, y_te):.3f}')
print(f'Test ROC-AUC : {roc_auc_score(y_te, xgb.predict_proba(X_te)[:, 1]):.3f}')

Best iteration found by early stopping: 252
Test accuracy: 0.848
Test ROC-AUC : 0.916


---
# 5️⃣ LightGBM — Light Gradient Boosting Machine

### Theory
LightGBM (Microsoft) is gradient boosting re-engineered for **speed and scale**:
- **Histogram-based splits:** continuous features are bucketed into ~255 bins → split search becomes dramatically faster and memory-light.
- **Leaf-wise growth** (vs XGBoost's classic level-wise): always splits the leaf with the largest loss reduction → deeper, more asymmetric trees, lower loss per tree (control overfitting with `num_leaves`).
- **GOSS** (Gradient-based One-Side Sampling): keeps all large-gradient samples, subsamples the easy ones.
- **EFB** (Exclusive Feature Bundling): merges mutually-exclusive sparse features.
- **Native categorical feature support** — no one-hot needed.

### Intuition
XGBoost carefully waters the whole garden row-by-row (level-wise). LightGBM walks straight to the thirstiest plant every time (leaf-wise) — faster progress per unit of work, but you must cap how wild the tree shape can get. On large datasets (millions of rows), LightGBM commonly trains **5–20× faster** with equal accuracy.

### Real-world use cases
- ⚡ **Large-scale ranking**: search & recommendation at Microsoft/Bing scale
- 📈 **High-frequency risk scoring** where retraining speed matters
- 🏪 **Retail demand forecasting** across millions of SKU-store pairs (e.g., M5 competition winner)
- 📱 Real-time **credit decisioning** APIs


In [6]:
# ==============================================================
# 5. LIGHTGBM — scikit-learn compatible API
# ==============================================================
from lightgbm import LGBMClassifier, early_stopping
import time

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,           # THE key complexity knob for leaf-wise growth (2^depth equivalent)
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)

t0 = time.time()
lgbm.fit(X_tr2, y_tr2,
         eval_set=[(X_val, y_val)],
         callbacks=[early_stopping(30, verbose=False)])
t_lgbm = time.time() - t0

print(f'Training time : {t_lgbm:.2f}s')
print(f'Test accuracy : {lgbm.score(X_te, y_te):.3f}')
print(f'Test ROC-AUC  : {roc_auc_score(y_te, lgbm.predict_proba(X_te)[:, 1]):.3f}')

# --- Head-to-head summary on the same dataset ---
from sklearn.metrics import accuracy_score
summary = pd.DataFrame({
    'Model':    ['RandomForest', 'XGBoost', 'LightGBM'],
    'Accuracy': [rf.score(X_te, y_te), xgb.score(X_te, y_te), lgbm.score(X_te, y_te)],
    'ROC-AUC':  [roc_auc_score(y_te, m.predict_proba(X_te)[:, 1]) for m in (rf, xgb, lgbm)],
}).round(4)
summary

Training time : 0.23s
Test accuracy : 0.843
Test ROC-AUC  : 0.913


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,Model,Accuracy,ROC-AUC
0,RandomForest,0.8450,0.9212
1,XGBoost,0.8475,0.9162
2,LightGBM,0.8425,0.9125


---
# 6️⃣ CNN — Convolutional Neural Networks *(Unstructured: Images / Audio)*

### Theory
A CNN learns hierarchies of visual features using three ideas:
1. **Convolution:** small learnable filters (e.g., 3×3) slide across the image computing dot products → produce *feature maps*. **Weight sharing** means the same filter detects a pattern anywhere in the image (translation invariance) with very few parameters.
2. **Pooling:** downsample feature maps (max-pooling keeps the strongest activation) → robustness to small shifts, fewer computations.
3. **Depth:** early layers learn edges → mid layers learn textures/shapes → deep layers learn objects (faces, wheels). Final dense layers classify.

Famous architectures: LeNet → AlexNet → VGG → ResNet (skip connections) → EfficientNet.

### Intuition
Instead of looking at all 1,000,000 pixels at once (what a dense network would do), a CNN looks through a small moving window, like your eye scanning a scene — "is there an edge here? a corner there?" — then combines local findings into a global understanding. It bakes in the physics of images: *nearby pixels are related; a cat is a cat wherever it sits in the frame.*

### Real-world use cases
- 🩻 **Medical imaging**: tumor detection in X-rays/MRI/CT
- 🚗 **Self-driving cars**: lane, pedestrian, sign detection
- 📱 **Face recognition** (phone unlock), photo tagging
- 🏭 **Manufacturing**: visual defect inspection on production lines
- 🎙️ **Audio**: speech commands via spectrogram images

> **Note:** scikit-learn has no convolutional layers (use PyTorch/TensorFlow for real CNNs). Below we demonstrate the *concept* on small images with sklearn's `MLPClassifier` (a fully-connected neural network) and manually show what a convolution filter does.


In [7]:
# ==============================================================
# 6a. What a convolution actually does — edge detection by hand
# ==============================================================
from sklearn.datasets import load_digits
from scipy.signal import convolve2d

digits = load_digits()                       # 1,797 tiny 8x8 grayscale digit images
img = digits.images[0]                       # one image of the digit '0'

# A classic vertical-edge filter (Sobel-like). CNNs LEARN thousands of such filters.
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])
feature_map = convolve2d(img, kernel, mode='valid')   # slide the filter over the image

fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(img, cmap='gray');         ax[0].set_title('Input image (8x8)')
ax[1].imshow(feature_map, cmap='gray'); ax[1].set_title('Feature map (vertical edges)')
for a in ax: a.axis('off')
plt.show()

In [8]:
# ==============================================================
# 6b. Neural network on images with scikit-learn (MLP)
#     — sklearn's closest tool; real CNNs need PyTorch/TensorFlow
# ==============================================================
from sklearn.neural_network import MLPClassifier

X_img = digits.data / 16.0                  # flatten 8x8 -> 64 pixels, scale to [0,1]
y_img = digits.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_img, y_img, test_size=0.2,
                                                  random_state=42, stratify=y_img)

mlp = MLPClassifier(hidden_layer_sizes=(128, 64),   # two hidden layers
                    activation='relu',
                    max_iter=500, random_state=42)
mlp.fit(X_tr_i, y_tr_i)
print(f'Digit classification accuracy (MLP): {mlp.score(X_te_i, y_te_i):.3f}')

# Show a few predictions
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
preds = mlp.predict(X_te_i[:6])
for ax, flat, p in zip(axes, X_te_i[:6], preds):
    ax.imshow(flat.reshape(8, 8), cmap='gray'); ax.set_title(f'pred: {p}'); ax.axis('off')
plt.show()

# Equivalent real-CNN skeleton (PyTorch, for reference — not run here):
#   nn.Sequential(nn.Conv2d(1, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
#                 nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
#                 nn.Flatten(), nn.Linear(64*..., 10))

Digit classification accuracy (MLP): 0.981


---
# 7️⃣ Transformers — BERT / GPT *(Unstructured: Text)*

### Theory
The Transformer (*"Attention Is All You Need"*, 2017) replaced recurrence with **self-attention**: every token computes a weighted view of every other token in the sequence:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^{T}}{\sqrt{d_k}}\right)V$$

- Each token emits a **Query** ("what am I looking for?"), a **Key** ("what do I contain?"), and a **Value** ("what do I contribute?").
- **Multi-head attention** runs several attention patterns in parallel (syntax, coreference, etc.).
- **Positional encodings** inject word order. Stacked layers + residual connections + layer norm → deep understanding.
- **BERT** = encoder-only, pre-trained by masked-word prediction → great for *understanding* (classification, NER, search). **GPT** = decoder-only, pre-trained by next-token prediction → great for *generation*. Both follow **pretrain on massive text → fine-tune on your task**.

### Intuition
When you read *"The bank raised interest rates"*, you instantly know *bank* means a financial institution — because your brain attends to *interest rates*. Self-attention gives every word that superpower: each word looks at the entire sentence and asks *"who is relevant to my meaning?"* — in parallel, at any distance. That solved the long-range-memory problem RNNs choked on, and it parallelizes beautifully on GPUs, enabling today's LLMs.

### Real-world use cases
- 💬 **Chat assistants & LLMs** (GPT, Claude) — generation, coding, reasoning
- 🔎 **Semantic search** & Google search ranking (BERT)
- 🌐 **Machine translation** (the original Transformer task)
- 😊 **Sentiment analysis**, document classification, named-entity recognition
- 🧬 Beyond text: **protein folding** (AlphaFold), vision (ViT), audio (Whisper)

> **Note:** scikit-learn cannot build a Transformer (use Hugging Face 🤗 `transformers`). Below we solve a text-classification task the *classical* sklearn way (TF-IDF + linear model) — the baseline every Transformer is compared against — plus a tiny numeric demo of attention weights.


In [9]:
# ==============================================================
# 7a. Self-attention in 15 lines of NumPy (the core equation)
# ==============================================================
tokens = ['the', 'bank', 'raised', 'interest', 'rates']
d = 4
rng = np.random.RandomState(0)
E  = rng.randn(len(tokens), d)               # pretend word embeddings
Wq, Wk, Wv = rng.randn(d, d), rng.randn(d, d), rng.randn(d, d)

Q, K, V = E @ Wq, E @ Wk, E @ Wv             # queries, keys, values
scores  = Q @ K.T / np.sqrt(d)               # similarity of every token with every token
attn    = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)   # softmax rows
output  = attn @ V                           # each token = weighted blend of all values

print('Attention weights (rows = who is attending, cols = attended to):')
print(pd.DataFrame(attn.round(2), index=tokens, columns=tokens))
# In a TRAINED model, "bank" would place high weight on "interest"/"rates".

Attention weights (rows = who is attending, cols = attended to):
           the  bank  raised  interest  rates
the       0.01  0.00    0.99      0.00   0.00
bank      0.43  0.01    0.54      0.01   0.00
raised    0.10  0.00    0.84      0.05   0.00
interest  0.20  0.07    0.53      0.15   0.05
rates     0.11  0.65    0.13      0.06   0.06


In [10]:
# ==============================================================
# 7b. Text classification the sklearn way — TF-IDF + Logistic Regression
#     (the classical baseline that Transformers later beat)
# ==============================================================
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

try:
    cats = ['sci.space', 'rec.sport.hockey', 'comp.graphics']
    train = fetch_20newsgroups(subset='train', categories=cats, remove=('headers','footers','quotes'))
    test  = fetch_20newsgroups(subset='test',  categories=cats, remove=('headers','footers','quotes'))
    texts_tr, y_tr_t, texts_te, y_te_t, names = train.data, train.target, test.data, test.target, train.target_names
except Exception:
    # Offline fallback: tiny handmade corpus
    texts_tr = ['rocket launch orbit nasa', 'hockey goal ice team win', 'render pixels 3d graphics gpu'] * 30
    y_tr_t   = [0, 1, 2] * 30
    texts_te = ['satellite orbit launch', 'ice hockey playoff goal', 'gpu shader 3d render']
    y_te_t   = [0, 1, 2]
    names    = ['space', 'hockey', 'graphics']

# TF-IDF: word counts weighted DOWN for words common in every document
vec = TfidfVectorizer(max_features=20000, stop_words='english')
Xtr = vec.fit_transform(texts_tr)             # sparse matrix: documents x vocabulary
Xte = vec.transform(texts_te)

text_clf = LogisticRegression(max_iter=2000).fit(Xtr, y_tr_t)
print(classification_report(y_te_t, text_clf.predict(Xte), target_names=names))

# Hugging Face equivalent (for reference — one line to use a real Transformer):
#   from transformers import pipeline
#   pipeline('sentiment-analysis')('I love this product!')

/usr/local/lib/python3.12/dist-packages/sklearn/datasets/_base.py:1519: UserWarning: Retry downloading from url: https://ndownloader.figshare.com/files/5975967
  warnings.warn(f"Retry downloading from url: {remote.url}")


              precision    recall  f1-score   support

       space       1.00      1.00      1.00         1
      hockey       1.00      1.00      1.00         1
    graphics       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



---
# 8️⃣ K-Means Clustering *(Unsupervised)*

### Theory
K-Means partitions unlabeled data into **K clusters** minimizing within-cluster variance (**inertia**):

$$J = \sum_{k=1}^{K}\sum_{x_i \in C_k} \lVert x_i - \mu_k \rVert^2$$

**Lloyd's algorithm:** 1) place K centroids (k-means++ smart init) → 2) **assign** each point to nearest centroid → 3) **update** each centroid to the mean of its points → repeat until stable. Guaranteed to converge (to a local optimum — hence `n_init` restarts).

**Choosing K:** the **Elbow method** (plot inertia vs K, look for the bend) and the **Silhouette score** (−1 to 1; how well each point fits its cluster vs the next-closest one).

**Limits:** assumes roughly spherical, similar-sized clusters; sensitive to scale (⚠️ standardize first!) and outliers.

### Intuition
Drop K magnets onto a table of iron filings. Filings snap to their nearest magnet; each magnet then slides to the center of its filings; the filings re-snap... after a few rounds everything settles into K natural groups. No labels were ever needed — structure emerges from distance alone.

### Real-world use cases
- 🛍️ **Customer segmentation** for targeted marketing (the #1 use)
- 🖼️ **Image compression / color quantization** (reduce an image to K colors)
- 📄 **Document grouping** & news topic bucketing
- 🏙️ **Facility placement**: where to put K warehouses/cell towers to minimize distances
- 🚨 **Anomaly detection**: points far from every centroid are suspicious


In [11]:
# ==============================================================
# 8. K-MEANS — scikit-learn, with Elbow + Silhouette for choosing K
# ==============================================================
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

# Unlabeled customer-like data with 4 hidden groups
X_u, _ = make_blobs(n_samples=900, centers=4, cluster_std=1.1, random_state=42)
X_u = StandardScaler().fit_transform(X_u)          # ALWAYS scale before K-Means

# --- Choose K ---
inertias, silhouettes, K_range = [], [], range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_u)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_u, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(K_range, inertias, 'o-');    ax[0].set_title('Elbow method');      ax[0].set_xlabel('K'); ax[0].set_ylabel('Inertia')
ax[1].plot(K_range, silhouettes, 'o-'); ax[1].set_title('Silhouette score');  ax[1].set_xlabel('K')
plt.tight_layout(); plt.show()

best_k = K_range[int(np.argmax(silhouettes))]
print(f'Best K by silhouette: {best_k}')

# --- Final clustering ---
km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X_u)
plt.figure(figsize=(5.5, 4.5))
plt.scatter(X_u[:, 0], X_u[:, 1], c=km.labels_, cmap='viridis', s=12, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, label='centroids')
plt.title(f'K-Means result (K={best_k})'); plt.legend(); plt.show()

Best K by silhouette: 4


---
# ✅ Algorithm Selection Cheat Sheet

| Situation | First choice | Why |
|---|---|---|
| Tabular + need interpretability | Linear / Logistic Regression | Coefficients tell the story |
| Tabular + best accuracy, medium data | Random Forest | Robust, near-zero tuning |
| Tabular + best accuracy, competitive | **XGBoost / LightGBM** | State of the art on tables |
| Tabular + millions of rows | LightGBM | Histogram + leaf-wise speed |
| Images / video / audio spectrograms | CNN | Local patterns + translation invariance |
| Text / sequences / language | Transformers | Self-attention, transfer learning |
| No labels, find groups | K-Means | Simple, fast, scalable |

**Universal workflow:** baseline (linear) → tree ensemble → tune → only then consider deep learning.

**Next notebook →** `3. Predictive Analytics Project`: everything combined in one end-to-end business project.
